<a href="https://colab.research.google.com/github/hridayrjain/BIS-Lab/blob/main/1BM24CS119_%20Library%20Book%20Arrangement%20Optimization_%20Parallel%20Cellular%20Algorithm%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import random

W_CAT, W_ACC = 1.0, 0.3

def neighbors(r, c):
    out = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < R and 0 <= nc < C:
            out.append((nr, nc))
    return out

def local_cost(grid, r, c):
    b = grid[r][c]
    mismatch = sum(1 for (nr, nc) in neighbors(r, c) if grid[nr][nc]["cat"] != b["cat"])
    access = b["demand"] * (abs(r - EYE_ROW) + c)
    return W_CAT * mismatch + W_ACC * access

def total_cost(grid):
    return sum(local_cost(grid, r, c) for r in range(R) for c in range(C))

def random_grid(books):
    b = books[:]
    random.shuffle(b)
    return [b[r * C:(r + 1) * C] for r in range(R)]

def pca_step(grid):
    claimed = set()
    cells = [(r, c) for r in range(R) for c in range(C)]
    random.shuffle(cells)
    for (r, c) in cells:
        if (r, c) in claimed:
            continue
        free = [n for n in neighbors(r, c) if n not in claimed]
        if not free:
            continue
        y = random.choice(free)

        affected = {(r, c), y}
        for n in neighbors(r, c) + neighbors(*y):
            affected.add(n)

        before = sum(local_cost(grid, a, b) for a, b in affected)
        grid[r][c], grid[y[0]][y[1]] = grid[y[0]][y[1]], grid[r][c]
        after = sum(local_cost(grid, a, b) for a, b in affected)

        if after < before:
            claimed.add((r, c))
            claimed.add(y)
        else: # undo swap
            grid[r][c], grid[y[0]][y[1]] = grid[y[0]][y[1]], grid[r][c]

def pca_library(books, P, max_iter, patience):
    population = [random_grid(books) for _ in range(P)]
    best_cost, best_grid, stale = float("inf"), None, 0
    for t in range(max_iter):
        for g in population:
            pca_step(g)
        gen_best = min(population, key=total_cost)
        cost = total_cost(gen_best)
        if cost < best_cost - 1e-9:
            best_cost, best_grid, stale = cost, [row[:] for row in gen_best], 0
        else:
            stale += 1
        if stale >= patience:
            break
    return best_grid, best_cost

def show(grid):
    w = max(len(b["title"]) for row in grid for b in row)
    for r, row in enumerate(grid):
        tag = " <- eye level" if r == EYE_ROW else ""
        print(" | ".join(f"{b['title']:<{w}} ({b['cat'][:3]})" for b in row) + tag)

def read_books(n):
    print(f"\nEnter {n} books, one per line as: title,category,popularity(1-10),availability(0-1)")
    print("Press Enter on an empty line to fill the rest with random sample books.")
    cats = ["Science", "History", "Fiction", "Tech"]
    books = []
    while len(books) < n:
        line = input(f"Book {len(books) + 1}: ").strip()
        if line == "":
            while len(books) < n:
                i = len(books) + 1
                books.append({"title": f"Book{i}", "cat": random.choice(cats),
                              "demand": random.randint(1, 10) * random.random()})
            break
        try:
            title, cat, pop, avail = [x.strip() for x in line.split(",")]
            books.append({"title": title, "cat": cat,
                          "demand": float(pop) * float(avail)})
        except ValueError:
            print(" Invalid format, try again.")
    return books

if __name__ == "__main__":
    R = int(input("Number of shelf levels (rows): "))
    C = int(input("Books per shelf (columns, column 0 = aisle): "))
    EYE_ROW = int(input(f"Eye-level shelf row (0 to {R - 1}): "))
    P = int(input("Number of arrangements in population (e.g. 5): "))
    max_iter = int(input("Max iterations (e.g. 200): "))

    books = read_books(R * C)
    start = random_grid(books)
    print("\nInitial random arrangement, cost =", round(total_cost(start), 2))
    show(start)

    best, cost = pca_library(books, P, max_iter, patience=20)
    print("\nOptimized arrangement, cost =", round(cost, 2))
    show(best)

Number of shelf levels (rows): 2
Books per shelf (columns, column 0 = aisle): 3
Eye-level shelf row (0 to 1): 0
Number of arrangements in population (e.g. 5): 5
Max iterations (e.g. 200): 100

Enter 6 books, one per line as: title,category,popularity(1-10),availability(0-1)
Press Enter on an empty line to fill the rest with random sample books.
Book 1: Physics101,Science,9,0.8
Book 2: Quantum,Science,4,1
Book 3: WorldWar2,History,7,0.6
Book 4: Rome,History,3,1
Book 5: Harry,Fiction,10,0.9
Book 6: 

Initial random arrangement, cost = 24.34
WorldWar2  (His) | Harry      (Fic) | Rome       (His) <- eye level
Physics101 (Sci) | Book6      (Tec) | Quantum    (Sci)

Optimized arrangement, cost = 19.2
Physics101 (Sci) | Harry      (Fic) | WorldWar2  (His) <- eye level
Quantum    (Sci) | Book6      (Tec) | Rome       (His)
